# Rust 123: CLI and Processes — Practice

Companion exercises for `23_cli-and-processes_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and every demonstration cell has an exercise, so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the `todo!()` bodies; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. Reading stdin: `input()` and Its Limits](#1-reading-stdin-input-and-its-limits)
- [2. The Desk: Build a CLI with a CLI](#2-the-desk-build-a-cli-with-a-cli)
- [3. Running a Program: `.output()` and `Output`](#3-running-a-program-output-and-output)
- [4. Exit Codes Are Data](#4-exit-codes-are-data)
- [5. stdout and stderr as Data](#5-stdout-and-stderr-as-data)
- [6. Piping stdin Into a Child](#6-piping-stdin-into-a-child)
- [7. Environment and Working Directory](#7-environment-and-working-directory)
- [8. `spawn`, `Child`, and Streaming stdout](#8-spawn-child-and-streaming-stdout)
- [9. Building a Helper, Not Just Running One](#9-building-a-helper-not-just-running-one)
- [10. Async: `tokio::process`](#10-async-tokioprocess)
- [11. clap: Typed Command-Line Interfaces](#11-clap-typed-command-line-interfaces)
- [12. Mixed Review](#12-mixed-review)

§1–§11 each have exercises for the demonstration cells of the example notebook (§0.1 is the *given* desk build, so §0's exercise is 0.2); §12 mixes corners. §12 of the example notebook is the trap catalogue.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from an earlier cell is visible in every cell below it — including `use` lines.
3. **Items are order-independent.** A struct or fn may be used above the cell that defines it.

> **Item names must be unique across the whole notebook.** Every cell shares one namespace; two functions with the same name are a redefinition error even in different sections. The exercise names here are unique by construction — do not introduce colliding helpers. The same rule covers `use` lines: the first cell that needs a name is the one that imports; later cells rely on persistence — a repeated `use` is E0252.

### The stub convention

Each exercise cell declares signatures with contract doc comments and a `// --- check (do not modify) ---` block of asserts. The body you must fill in is `todo!()`, which compiles but **panics** when executed. Replace it and re-run; an `Exercise N.M passed` line means you are done.

### Four notes specific to this chapter

- **The desk builds once, in §0.1.** The given `desk_build()` writes the stub source into `scratch23/` and compiles it — after §0.1 the exe persists for the whole session, exactly like the kernel's namespace. If a check fails with `NotFound` on `./scratch23/docker-stub.exe`, restart and run from the top.
- **Async stubs are driven by the check's runtime** — which the check scopes inside a block so it drops before the check ends. An `async fn` stub is awaited inside the check's `block_on`, and **nothing prints inside `block_on`** — compute inside, assert and print after (§10/§12 of the example notebook).
- **No wall-clock asserts.** Cells that involve waiting return behavioural values; the sleeps carry enormous margins (a 300 ms probe against a 30 s sleeper). Never assert on a duration.
- **Captures are owned.** Text coming out of a child comes back as `String` (`.into_owned()`), never a borrowed `Cow<str>` — the kernel refuses to persist one (§5/§12 of the example notebook).

### Expect warnings until you finish

Three warning families are normal and self-cleaning:

- `unused variable` — every unsolved stub whose parameters are untouched;
- `unused import` — the `use` lines serving a stub you have not solved yet;
- **stub-induced dead code** (`never used` / `never constructed`) — given helpers that only a *solution* calls.

Anything outside these families in your solve is a real smell.


In [ ]:
// 0.1 Given: the crate cell and the desk build. Run once, run first.
:dep clap = { version = "4", features = ["derive"] }
:dep tokio = { version = "1", features = ["full"] }

use std::process::Command;

/// Write the docker-stub source into scratch23/ and compile it with rustc.
/// Given: the whole notebook tests against this desk. (Source identical to §2.1
/// of the example notebook; see there for the line-by-line reading.)
fn desk_build() {
    let stub_lines = [
        "struct Container { id: &'static str, image: &'static str, names: &'static str, status: &'static str, ports: &'static str }",
        "const CONTAINERS: [Container; 3] = [",
        "    Container { id: \"3f1a9c2e7b01\", image: \"redis:7-alpine\", names: \"md-cache\", status: \"Up 2 hours\", ports: \"0.0.0.0:6379->6379/tcp\" },",
        "    Container { id: \"8d44b0aa19ce\", image: \"postgres:16\", names: \"md-store\", status: \"Up 2 hours\", ports: \"0.0.0.0:5432->5432/tcp\" },",
        "    Container { id: \"c07e55d3f8a2\", image: \"ghcr.io/acme/feed:3\", names: \"md-feed\", status: \"Exited (137) 5 minutes ago\", ports: \"\" },",
        "];",
        "fn main() { let args: Vec<String> = std::env::args().skip(1).collect();",
        "    match args.first().map(|s| s.as_str()).unwrap_or(\"\") {",
        "        \"ps\" => ps(&args),",
        "        \"version\" => println!(\"Docker version {}, build f4e3f4e\",",
        "            std::env::var(\"STUB_VERSION\").unwrap_or_else(|_| \"23.0.0-stub\".into())),",
        "        \"logs\" => logs(args.get(1).map(|s| s.as_str()).unwrap_or(\"\")),",
        "        \"sleep\" => sleep(args.get(1).and_then(|s| s.parse().ok()).unwrap_or(1.0)),",
        "        other => { eprintln!(\"docker: 'docker {}' is not a docker-stub command.\", other);",
        "            eprintln!(\"usage: docker-stub <ps|version|logs|sleep> [args]\");",
        "            std::process::exit(2); }",
        "    } }",
        "fn logs(id: &str) {",
        "    println!(\"[{id}] feed: connected to upstream\");",
        "    println!(\"[{id}] feed: 5120 ticks replayed\");",
        "    eprintln!(\"[{id}] feed: upstream disconnect, reconnecting...\");",
        "    println!(\"[{id}] feed: resumed at seq 5121\"); }",
        "fn sleep(secs: f64) { let mut done = 0.0;",
        "    while done < secs { std::thread::sleep(std::time::Duration::from_millis(100)); done += 0.1; }",
        "    println!(\"slept {secs}s\"); }",
        "fn ps(args: &[String]) {",
        "    let mut json_mode = false;",
        "    for a in args { if a == \"--format\" { json_mode = true; } }",
        "    if json_mode { for c in &CONTAINERS {",
        "        println!(\"{{\\\"ID\\\":\\\"{}\\\",\\\"Image\\\":\\\"{}\\\",\\\"Names\\\":\\\"{}\\\",\\\"State\\\":\\\"{}\\\",\\\"Status\\\":\\\"{}\\\",\\\"Ports\\\":\\\"{}\\\"}}\",",
        "            c.id, c.image, c.names,",
        "            if c.status.starts_with(\"Up\") { \"running\" } else { \"exited\" }, c.status, c.ports); }",
        "    } else {",
        "        println!(\"{:<12} {:<26} {:<10} {}\", \"CONTAINER ID\", \"IMAGE\", \"NAMES\", \"STATUS\");",
        "        for c in &CONTAINERS { println!(\"{:<12} {:<26} {:<10} {}\", c.id, c.image, c.names, c.status); } } }",
    ];
    std::fs::create_dir_all("scratch23").unwrap();
    std::fs::write("scratch23/docker-stub.rs", stub_lines.join("\n")).unwrap();
    let build = Command::new("rustc")
        .args(["--edition", "2024", "-O", "-o", "scratch23/docker-stub.exe", "scratch23/docker-stub.rs"])
        .output().unwrap();
    assert!(build.status.success(), "stub build failed: {}", String::from_utf8_lossy(&build.stderr));
}

desk_build();
println!("desk ready: scratch23/docker-stub.exe");


### Exercise 0.2 — Toolchain smoke

**Goal**: prove you can capture a child's stdout before touching the stub.

**Objectives**

- return the trimmed first line of `rustc --version` as an owned `String`
- keep the signature unchanged

**See**: §0.2 of the example notebook.

**Hint**: `Command::new("rustc").arg("--version").output()` — then `from_utf8_lossy`, and the one method this notebook insists on before returning text.


In [ ]:
// 0.2 Exercise: capture rustc --version as the chapter's first receipt.

/// Return `rustc --version`'s stdout, trimmed, as an owned String.
fn rustc_version_line() -> String {
    todo!()
}

// --- check (do not modify) ---
let v = rustc_version_line();
assert!(v.starts_with("rustc"), "got: {v}");
assert!(v.contains("1.98"), "got: {v}");
println!("Exercise 0.2 passed");


## 1. Reading stdin: `input()` and Its Limits

EOF is `Ok(0)`; a bad number is a data error carrying the offending text. The kernel's own stdin is dead (§0 of the example notebook) — readers are parameters, and children are the place where stdin moves for real.

**See**: §1 of the example notebook.


In [ ]:
// 1.1 Exercise: the typed input() pattern over any BufRead.
use std::io::BufRead;

/// Read one line from `input` and parse it as f64.
/// EOF (0 bytes read) is Err("end of input (EOF)"); a failed parse is
/// Err containing the offending trimmed text.
fn read_number(input: &mut dyn BufRead) -> Result<f64, String> {
    todo!()
}

// --- check (do not modify) ---
let mut good = "4125.5\n".as_bytes();
assert_eq!(read_number(&mut good), Ok(4125.5));
let mut bad = "not-a-number\n".as_bytes();
let err = read_number(&mut bad).unwrap_err();
assert!(err.contains("not-a-number"), "got: {err}");
let mut empty = "".as_bytes();
let eof = read_number(&mut empty).unwrap_err();
assert!(eof.contains("EOF"), "got: {eof}");
println!("Exercise 1.1 passed");


### Exercise 1.2 — Bytes into a child's stdin

**Goal**: write text into a child's stdin and read its answer.

**Info**: the write side must drop before you collect, or the child never sees EOF.

**Given**: `use std::io::Write;` and `use std::process::Stdio;` arrived in this cell.

**Objectives**

- spawn `cmd /C findstr typed` with piped stdin and stdout
- write `typed line reaches a child\n` into the pipe, then close it
- return the child's stdout as an owned `String`
- keep the signature unchanged

**See**: §1.2 of the example notebook.

**Hint**: `child.stdin.take()` inside a block; `wait_with_output()` after it.


In [ ]:
// 1.2 Exercise: the stdin receipt, written by you.
use std::io::Write;
use std::process::Stdio;

/// Feed the given text to `cmd /C findstr typed` through a piped stdin and
/// return the filtered stdout as an owned String.
fn feed_child_stdin(text: &[u8]) -> String {
    todo!()
}

// --- check (do not modify) ---
let got = feed_child_stdin(b"typed line reaches a child\n");
assert!(got.contains("typed line reaches a child"), "got: {got:?}");
println!("Exercise 1.2 passed");


## 2. The Desk: Build a CLI with a CLI

The desk is built (§0.1). These exercises drive it: the JSON-lines listing, then a general runner that reports code and both channels.

**See**: §2 of the example notebook.


In [ ]:
// 2.1 Exercise: the ps --format json listing as lines.

/// Run `docker-stub ps --format {{json .}}` and return its stdout split into
/// lines (owned Strings, `\r`/`\n` stripped by `lines()`).
fn ps_json_lines() -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
let lines = ps_json_lines();
assert_eq!(lines.len(), 3, "got: {lines:?}");
assert!(lines[0].contains("\"Names\":\"md-cache\""), "got: {}", lines[0]);
assert!(lines[2].contains("md-feed"), "got: {}", lines[2]);
println!("Exercise 2.1 passed");


### Exercise 2.2 — The general runner

**Goal**: one function that runs any stub argv and reports everything.

**Objectives**

- return `(exit_code, stdout, stderr)` with both texts owned
- keep the signature unchanged

**See**: §2.2 of the example notebook.

**Hint**: `.output()` gives all three at once; `status.code().unwrap_or(-1)` keeps the tuple `i32`.


In [ ]:
// 2.2 Exercise: code + stdout + stderr in one call.

/// Run `./scratch23/docker-stub.exe` with `argv` and return
/// (exit code, stdout as owned String, stderr as owned String).
fn run_stub(argv: &[&str]) -> (i32, String, String) {
    todo!()
}

// --- check (do not modify) ---
let (code, out, err) = run_stub(&["version"]);
assert_eq!(code, 0);
assert!(out.contains("Docker version"), "out: {out:?}");
let (code2, out2, err2) = run_stub(&["bogus"]);
assert_eq!(code2, 2);
assert!(out2.is_empty(), "out: {out2:?}");
assert!(err2.contains("not a docker-stub command"), "err: {err2:?}");
println!("Exercise 2.2 passed");


## 3. Running a Program: `.output()` and `Output`

The `Err` arm is about launching, not about the child's opinion — and the success path is three plain fields.

**See**: §3 of the example notebook.


In [ ]:
// 3.1 Exercise: Output's fields as values.

/// Capture `docker-stub version` with .output() and return its stdout length
/// in bytes.
fn output_bytes_len() -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(output_bytes_len(), 42);
println!("Exercise 3.1 passed");


### Exercise 3.2 — The launch failure's kind

**Goal**: classify a missing program without panicking.

**Objectives**

- attempt to run a program that does not exist and return its `ErrorKind`
- keep the signature unchanged

**See**: §3.2 of the example notebook.

**Hint**: the `Err` arm of `.output()`; `e.kind()` — no `unwrap` anywhere on that path.


In [ ]:
// 3.2 Exercise: NotFound as a value you can assert on.

/// Attempt to run "definitely-missing-p23-9q" and return the io::ErrorKind
/// of the launch failure.
fn missing_program_kind() -> std::io::ErrorKind {
    todo!()
}

// --- check (do not modify) ---
assert!(matches!(missing_program_kind(), std::io::ErrorKind::NotFound));
println!("Exercise 3.2 passed");


## 4. Exit Codes Are Data

`success()` for the bool, `code()` for the number — the stub speaks 0 and 2.

**See**: §4 of the example notebook.


In [ ]:
// 4.1 Exercise: the code trichotomy, sampled.

/// Run `docker-stub ps` and `docker-stub bogus`; return their exit codes as
/// (ps_code, bogus_code).
fn exit_code_pair() -> (i32, i32) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(exit_code_pair(), (0, 2));
println!("Exercise 4.1 passed");


### Exercise 4.2 — `status()`: the code without the bytes

**Goal**: use the inherited-streams runner and read only the verdict.

**Objectives**

- run `docker-stub version` with `.status()` and return `success()`
- keep the signature unchanged

**See**: §4.2 of the example notebook.

**Hint**: the version line will print into the cell unasked — that is `inherit` doing its job.


In [ ]:
// 4.2 Exercise: status-only run.

/// Run `docker-stub version` with .status() and return whether it succeeded.
fn status_success() -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(status_success());
println!("Exercise 4.2 passed");


## 5. stdout and stderr as Data

Two channels, two counts; the text arrives as bytes and leaves owned.

**See**: §5 of the example notebook.


In [ ]:
// 5.1 Exercise: count both channels of the logs command.

/// Run `docker-stub logs 3f1a9c2e7b01` and return
/// (stdout line count, stderr line count).
fn channel_line_counts() -> (usize, usize) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(channel_line_counts(), (3, 1));
println!("Exercise 5.1 passed");


### Exercise 5.2 — The capture chain

**Goal**: bytes → lossy text → first line, ending owned.

**Objectives**

- return the FIRST line of `ps --format {{json .}}` as an owned `String`
- keep the signature unchanged

**See**: §5.2 of the example notebook.

**Hint**: `.lines().next()` — and remember the method that makes the `Cow` an owned `String`.


In [ ]:
// 5.2 Exercise: first JSON line, owned.

/// Return the first line of `docker-stub ps --format {{json .}}`'s stdout,
/// as an owned String (no trailing newline).
fn first_json_line() -> String {
    todo!()
}

// --- check (do not modify) ---
let line = first_json_line();
assert!(line.contains("md-cache"), "got: {line}");
assert!(line.ends_with('}'), "got: {line}");
println!("Exercise 5.2 passed");


## 6. Piping stdin Into a Child

The producer→filter→consumer pipeline, parameterised: any word, any lines.

**See**: §6 of the example notebook.


In [ ]:
// 6.1 Exercise: the generalised findstr pipeline.

/// Feed `lines` (each a separate line) through `cmd /C findstr <word>` via a
/// piped stdin and return the filtered stdout as an owned String.
fn feed_filter(word: &str, lines: &[&str]) -> String {
    todo!()
}

// --- check (do not modify) ---
let hits = feed_filter("AAPL", &["MSFT 231.75", "AAPL 4125.50", "NVDA 187.20"]);
assert!(hits.contains("AAPL 4125.50"), "got: {hits:?}");
assert!(!hits.contains("MSFT"), "got: {hits:?}");
let none = feed_filter("ZZZZ", &["MSFT 231.75", "AAPL 4125.50"]);
assert!(none.trim().is_empty(), "got: {none:?}");
println!("Exercise 6.1 passed");


## 7. Environment and Working Directory

One variable for one child; one directory for one child.

**See**: §7 of the example notebook.


In [ ]:
// 7.1 Exercise: STUB_VERSION from the parent's side.

/// Run `docker-stub version` with STUB_VERSION set to `version` and return
/// the stdout as an owned String.
fn version_under_env(version: &str) -> String {
    todo!()
}

// --- check (do not modify) ---
let out = version_under_env("99.9.9-p23");
assert!(out.contains("99.9.9-p23"), "got: {out}");
println!("Exercise 7.1 passed");


### Exercise 7.2 — The child's pwd

**Goal**: move a child's working directory and observe it from inside.

**Objectives**

- run `cmd /C cd` with `current_dir(dir)` and return the child's cwd as a trimmed owned `String`
- keep the signature unchanged

**See**: §7.2 of the example notebook.

**Hint**: the builder method is `.current_dir(..)`; `trim()` before returning keeps the receipt clean.


In [ ]:
// 7.2 Exercise: current_dir, observed by the child.

/// Run `cmd /C cd` with the child's working directory set to `dir`; return
/// the printed cwd as a trimmed owned String.
fn child_cwd(dir: &str) -> String {
    todo!()
}

// --- check (do not modify) ---
let got = child_cwd("C:/Windows/Temp");
assert_eq!(got, "C:\\Windows\\Temp", "got: {got:?}");
println!("Exercise 7.2 passed");


## 8. `spawn`, `Child`, and Streaming stdout

A `Child` is a resource: take its pipes, stream its lines, probe it, kill it, reap it — and spell the program path.

**See**: §8 of the example notebook.


In [ ]:
// 8.1 Exercise: stream a child's stdout as it writes.
use std::io::BufReader;

/// Spawn `docker-stub logs <id>` with piped stdout, collect the streamed
/// lines (BufReader + lines()) as owned Strings, wait, and return them.
fn stream_log_lines(id: &str) -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
let lines = stream_log_lines("3f1a9c2e7b01");
assert_eq!(lines.len(), 3, "got: {lines:?}");
assert!(lines[0].contains("connected to upstream"), "got: {}", lines[0]);
println!("Exercise 8.1 passed");


### Exercise 8.2 — Probe, kill, reap

**Goal**: the full lifecycle against a long-running child.

**Info**: `sleep 30` runs for thirty seconds; a 300 ms probe cannot race it.

**Objectives**

- spawn `docker-stub sleep 30`, verify at ~300 ms that it is still running (assert this inside the body)
- `kill()`, then `wait()`, and return whether the final status was NOT a success
- keep the signature unchanged

**See**: §8.2 of the example notebook.

**Hint**: `try_wait()` returns `Ok(None)` while it lives; on this platform the killed child's code is `Some(1)`.


In [ ]:
// 8.2 Exercise: the lifecycle receipt.

/// Spawn `docker-stub sleep 30`, assert it is alive at ~300 ms, kill it,
/// wait for the exit, and return `!status.success()`.
fn kill_after_start() -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(kill_after_start());
println!("Exercise 8.2 passed");


### Exercise 8.3 — The lookup rule

**Goal**: prove the bare-name rejection yourself.

**Objectives**

- attempt to spawn `docker-stub.exe` (no `./`) and return whether it failed
- keep the signature unchanged

**See**: §8.3 of the example notebook.

**Hint**: `.output()`'s `Err` arm; `.is_err()` is the whole answer.


In [ ]:
// 8.3 Exercise: bare name is NotFound even in its own directory.

/// Attempt to run "docker-stub.exe" (bare name) and return whether the
/// launch failed.
fn bare_name_fails() -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(bare_name_fails());
println!("Exercise 8.3 passed");


## 9. Building a Helper, Not Just Running One

The compose chain, end to end — no timing asserts, only behaviour.

**See**: §9 of the example notebook.


In [ ]:
// 9.1 Exercise: write -> rustc -> run, verified behaviourally.

/// Recompile the stub (rustc on scratch23/docker-stub.rs, source still on
/// disk from §0.1), assert the build succeeded, run `docker-stub version`,
/// and return whether its stdout mentions "Docker version".
fn rebuild_desk_works() -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(rebuild_desk_works());
println!("Exercise 9.1 passed");


## 10. Async: `tokio::process`

The async stubs are awaited inside the check's `block_on` — nothing prints inside it, and each check scopes its runtime so it drops before the check ends. Compute, return owned data, let the check assert and print.

**See**: §10 of the example notebook.


In [ ]:
// 10.1 Exercise: .output().await over the desk.

/// Async twin of 2.1: run `docker-stub ps --format {{json .}}` with tokio's
/// Command and return the number of JSON lines in its stdout.
async fn ps_json_line_count() -> usize {
    todo!()
}

// --- check (do not modify) ---
let n = {
    let rt = tokio::runtime::Runtime::new().unwrap();
    rt.block_on(ps_json_line_count())
};
assert_eq!(n, 3);
println!("Exercise 10.1 passed");


### Exercise 10.2 — Async line streaming

**Goal**: drain a child's stdout through `AsyncBufReadExt::next_line()`.

**Given**: `use tokio::io::AsyncBufReadExt;` arrived in this cell; the reader struct is spelled `tokio::io::BufReader` (std's `BufReader` owns the name from §8.1). Pipe **stdout only**, and keep the check's runtime scoped inside a block — both are this kernel's proven shape (receipts in the example notebook's §12).

**Objectives**

- async-spawn `docker-stub logs 3f1a9c2e7b01` with stdout piped (stderr stays inherited)
- drain every line, await `wait()`, and return `(line_count, status.success())`
- keep the signature unchanged

**See**: §10.2 of the example notebook.

**Hint**: `while let Some(_l) = lines.next_line().await.unwrap()`.


In [ ]:
// 10.2 Exercise: the async drain.
use tokio::io::AsyncBufReadExt;

/// Async-spawn `docker-stub logs <id>`, pipe stdout only (stderr stays
/// inherited), stream it line by line with next_line().await, wait, and
/// return (lines streamed, success).
async fn stream_log_receipt(id: &str) -> (usize, bool) {
    todo!()
}

// --- check (do not modify) ---
let (n, ok) = {
    let rt = tokio::runtime::Runtime::new().unwrap();
    rt.block_on(stream_log_receipt("3f1a9c2e7b01"))
};
assert_eq!(n, 3);
assert!(ok);
println!("Exercise 10.2 passed");


### Exercise 10.3 — Timeout and `kill_on_drop`

**Goal**: don't wait forever, don't leak either.

**Objectives**

- async-spawn `docker-stub sleep 30` with `kill_on_drop(true)`
- wrap `wait()` in a 400 ms `tokio::time::timeout`; on miss, kill explicitly
- return whether the child ended up NOT successfully completed
- keep the signature unchanged

**See**: §10.3 of the example notebook.

**Hint**: the `match` has two arms — `Ok(res)` (completed) and `Err(_elapsed)` (timed out); only one of them fires for a 30 s sleeper.


In [ ]:
// 10.3 Exercise: the timed-out fleet member.

/// Async-spawn `docker-stub sleep 30` with kill_on_drop(true); wait at most
/// 400 ms (tokio::time::timeout); on miss kill explicitly. Return true when
/// the child did NOT complete successfully.
async fn timeout_kills_stub() -> bool {
    todo!()
}

// --- check (do not modify) ---
let killed = {
    let rt = tokio::runtime::Runtime::new().unwrap();
    rt.block_on(timeout_kills_stub())
};
assert!(killed);
println!("Exercise 10.3 passed");


## 11. clap: Typed Command-Line Interfaces

The interface is given as a type (above the check marker, like every given); your job is the parsing calls. Remember the annotation rule for anything you bind at cell top level.

**See**: §11 of the example notebook.


In [ ]:
// 11.1 Exercise: parse argv into the given interface. (Given: Ps + Fmt.)
use clap::{Parser, ValueEnum};

/// List the stub fleet, docker-ps style. Given: the exercise parses into it.
#[derive(Parser, Debug)]
#[command(name = "stub-ps", about = "list the stub containers")]
struct Ps {
    /// output format: table or json lines
    #[arg(short, long, value_enum, default_value = "table")]
    format: Fmt,
    /// limit the listing to the first N containers
    #[arg(long, default_value = "12")]
    last: u32,
}

#[derive(ValueEnum, Clone, Debug)]
enum Fmt {
    Table,
    Json,
}

/// Parse `argv` (argv[0] is the program name) into the given Ps interface
/// and return (format Debug string, last) -- e.g. ("Json", 5).
fn parse_stub_ps(argv: &[&str]) -> (String, u32) {
    todo!()
}

// --- check (do not modify) ---
let (fmt, last) = parse_stub_ps(&["stub-ps", "--format", "json", "--last", "5"]);
assert_eq!(fmt, "Json");
assert_eq!(last, 5);
let (fmt_d, last_d) = parse_stub_ps(&["stub-ps"]);
assert_eq!(fmt_d, "Table");
assert_eq!(last_d, 12);
println!("Exercise 11.1 passed");


### Exercise 11.2 — The rejection, as a value

**Goal**: catch clap's error with `try_parse_from` and return its first line.

**Objectives**

- attempt `["stub-ps", "--format", "yaml"]` and return the first line of the error's Display
- keep the signature unchanged

**See**: §11.2 of the example notebook.

**Hint**: the expected text begins `error: invalid value 'yaml'` — `e.to_string().lines().next()`.


In [ ]:
// 11.2 Exercise: try_parse_from without the process exit.

/// Try to parse `["stub-ps", "--format", "yaml"]` into Ps; return the first
/// line of clap's error Display (empty string if it unexpectedly parses).
fn reject_bad_format() -> String {
    todo!()
}

// --- check (do not modify) ---
let msg = reject_bad_format();
assert!(msg.starts_with("error: invalid value 'yaml'"), "got: {msg}");
println!("Exercise 11.2 passed");


### Exercise 11.3 — Help, for free

**Goal**: inspect the generated help as data.

**Objectives**

- render `Ps`'s help via `CommandFactory` and return whether it mentions the given flag
- keep the signature unchanged

**See**: §11.3 of the example notebook.

**Hint**: `Ps::command().render_help().to_string()`; the `use clap::CommandFactory;` arrived in this cell.


In [ ]:
// 11.3 Exercise: the generated help, inspected.
use clap::CommandFactory;

/// Return whether Ps's rendered help mentions `flag` (e.g. "--format").
fn help_mentions(flag: &str) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(help_mentions("--format"));
assert!(help_mentions("--last"));
assert!(!help_mentions("--nonexistent"));
println!("Exercise 11.3 passed");


## 12. Mixed Review

Three corners in one place: output shapes, exit-code discipline, and the async tour.

**See**: §12 of the example notebook is the trap catalogue — these exercises avoid exactly those traps.


In [ ]:
// 12.1 Exercise: the two output shapes, counted.

/// Run `docker-stub ps` (table mode) and `docker-stub ps --format {{json .}}`;
/// return (table line count, json line count).
fn output_shape_pair() -> (usize, usize) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(output_shape_pair(), (4, 3)); // table: header + 3 rows
println!("Exercise 12.1 passed");


### Exercise 12.2 — Exit-code discipline

**Goal**: the usage error and the success, sampled back to back.

**Objectives**

- run `bogus` then `version`; return `(bogus_code, version_code)`
- keep the signature unchanged

**See**: §4 of the example notebook.

**Hint**: `run_stub` from §2.2 persists — or `.output()` again; `code()` is an `Option`.


In [ ]:
// 12.2 Exercise: 2 then 0.

/// Run `docker-stub bogus` and then `docker-stub version`; return their exit
/// codes as (bogus_code, version_code).
fn exit_code_discipline() -> (i32, i32) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(exit_code_discipline(), (2, 0));
println!("Exercise 12.2 passed");


### Exercise 12.3 — The async tour

**Goal**: consolidate §10 — two calls, one runtime, nothing printed inside.

**Objectives**

- inside one `block_on`-driven body: async-run `version` (piped stdout) and `ps --format {{json .}}`
- return `(version stdout starts with "Docker version", json line count)`
- keep the signature unchanged

**See**: §10.1 of the example notebook.

**Hint**: two `.output().await`s, one owned `String` each — then the tuple.


In [ ]:
// 12.3 Exercise: the async tour receipt.

/// Async-run `docker-stub version` and `docker-stub ps --format {{json .}}`;
/// return (version stdout starts with "Docker version", json line count).
async fn async_tour_receipt() -> (bool, usize) {
    todo!()
}

// --- check (do not modify) ---
let (starts, n) = {
    let rt = tokio::runtime::Runtime::new().unwrap();
    rt.block_on(async_tour_receipt())
};
assert!(starts);
assert_eq!(n, 3);
println!("Exercise 12.3 passed");


### What Comes Next

Chapter 24 parses what this chapter captured: the JSON lines of §2/§5/§10 meet `serde_json`, regex, and CSV. Chapter 25 turns this notebook's checks into `#[test]` discipline — and chapter 26 opens the macro machinery `#[derive(Parser)]` has been using since §11.

Carry forward: spawn by `./`-path, capture with `.output()`, treat `Child` as a resource, decode bytes once and own them, and let clap's derive own the argv grammar.
